# Transformer Architecture

In this chapter, we explain in detail a fundamental mechanism behind LLMs - the **transformer** neural network architecture. Transformers are an innovation in the design of neural networks that enabled us to train much more capable language models than before.

## Intuition for the Attention mechanism

Let's imagine you are an LLM and your task is to translate __Ich bin mit dem Fahrrad hingefahren__ into English.

Which German words should you pay attention to, as you go on writing your English translation?

<div style="text-align: center; margin: 30px 0;">
  <img src="graphics/transformers_attention_intuition.svg" style="max-width: 800px; max-height: 500px; border: 2px solid #ddd; border-radius: 8px; padding: 5px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
  <div style="margin-top: 10px; font-style: italic; color: #666; font-size: 0.9em;">
    Transformer attention mechanism intuition
  </div>
</div>

For our translation LLM to work well, it needs to have

* full access to each of the individual words of the original German sentence
* mechanisms for giving different attention/weight to each of these words

*( *spoiler*: Both of these things are missing in Recurrent Neural Networks, which puts them at a huge disadvantage.)* 

## Word / token embeddings and dot products

We can not directly input strings into a neural network. Instead, we encode each word or part of word (_token_) as a _multi-dimensional vector_. 

<div style="text-align: center; margin: 30px 0;">
  <img src="graphics/transformers_embedding.svg" style="max-width: 700px; max-height: 400px; border: 2px solid #ddd; border-radius: 8px; padding: 5px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
  <div style="margin-top: 10px; font-style: italic; color: #666; font-size: 0.9em;">
    Word embedding visualization
  </div>
</div>

We expect words/tokens that are similar to each other to have similar embedding vectors

<div style="text-align: center; margin: 30px 0;">
  <img src="graphics/transformers_embedding2.svg" style="max-width: 700px; max-height: 400px; border: 2px solid #ddd; border-radius: 8px; padding: 5px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
  <div style="margin-top: 10px; font-style: italic; color: #666; font-size: 0.9em;">
    Token and position embeddings
  </div>
</div>

By taking the dot product between two vectors, we can calculate their similarity.

In [ ]:
import numpy as np

In [ ]:
# identical vectors have a dot product of one
v1 = np.array([0, 0, 1])
v2 = np.array([0, 0, 1])

np.dot(v1, v2)

In [ ]:
# vectors similar to each other have a dot product near 1
v1 = np.array([0.01, 0.01, 0.98])
v2 = np.array([0, 0, 1])

np.dot(v1, v2)

In [ ]:
# vectors different from each other have a dot product near 0
v1 = np.array([0, 0, 1])
v2 = np.array([1, 0, 0])

np.dot(v1, v2)

In [ ]:
# vectors different from each other have a dot product near 0
v1 = np.array([0, 0, 1])
v2 = np.array([1, 1, 0.01])

np.dot(v1, v2)

## The attention mechanism

Let's turn again to the problem of translating from German to English and walk through the attention mechanism.


### Defining the embeddings

In [ ]:
# we first need the embeddings for all the words in our German sentence
keys = {
    "Ich": np.array([1.0, 0.0, 0.0]),
    "bin": np.array([0.2, 0.8, 0.0]),
    "mit": np.array([0.0, 0.1, 0.9]),
    "dem": np.array([0.5, 0.0, 0.5]),
    "Fahrrad": np.array([0.4, 0.1, 0.5]),
    "gefahren": np.array([0.3, 0.7, 0.0]),
}
keys


In [ ]:

# let's imagine that we have already translated the first word "I"
# here is the embedding for that word
query = np.array([1.0, 0.0, 0.0])

Here is what we have so far. The names "query" and "keys" come from information retrieval.

<div style="text-align: center; margin: 30px 0;">
  <img src="graphics/transformers_attention_details1.svg" style="max-width: 800px; max-height: 600px; border: 2px solid #ddd; border-radius: 8px; padding: 5px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
  <div style="margin-top: 10px; font-style: italic; color: #666; font-size: 0.9em;">
    Attention mechanism details (step 1)
  </div>
</div>

### Multiplying the embeddings with a learnable weight matrix

We don't use the embeddings directly, but instead transform them by multiplying with some learnable matrix. 
We'll see later why that is necessary

In [ ]:
# define the weight matrix to transform the keys
# its shape is (embedding_dimension x embedding_dimensions), i.e. 3x3 for our example
M_k = np.array(
   [[0, 1, 0],
    [1, 0, 0],
    [0, 0, 1]] 
)

# we use a different matrix to transform the query
M_q = np.array(
   [[1, 0, 0],
    [0, 1, 0],
    [0, 0, 1]] 
)

# multiply each key embedding with the key matrix
keys = {word: np.dot(embedding, M_k) for word, embedding in keys.items()}

# multiply the query embedding with the query matrix
query = np.dot(query, M_q)

keys

Here is what we have so far, visually

<div style="text-align: center; margin: 30px 0;">
  <img src="graphics/transformers_attention_details2.svg" style="max-width: 800px; max-height: 600px; border: 2px solid #ddd; border-radius: 8px; padding: 5px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
  <div style="margin-top: 10px; font-style: italic; color: #666; font-size: 0.9em;">
    Attention mechanism details (step 2)
  </div>
</div>

### Calculating the attention

We now take the dot product between each key with the query. This calculates for each key how well it "matches" the query.

_(Perhaps now it becomes clear why we needed to multiply with the matrices above. If we had not done it, "I" would have the
highest similarity with "Ich" rather than "bin" und "hingefahren")._

In [ ]:
attention = {word: np.dot(key, query) for word, key in keys.items()}
attention

In [ ]:
# we make sure that the values sum up to 1
# (usually we would use a softmax for that, but let's keep it simple here)
attention = {word: value / sum(attention.values()) for word, value in attention.items()} 
attention

<div style="text-align: center; margin: 30px 0;">
  <img src="graphics/transformers_attention_details3.svg" style="max-width: 800px; max-height: 600px; border: 2px solid #ddd; border-radius: 8px; padding: 5px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
  <div style="margin-top: 10px; font-style: italic; color: #666; font-size: 0.9em;">
    Attention mechanism details (step 3)
  </div>
</div>

### Calculate weighted summary of input sentence

We then take the weighted sum of the German embeddings using the attention vector.
This gives us a "summary" of the input sentence, with most weights put on the important words.

(In practice, we apply another learnable weight matrix onto the keys before multiplying them, but let's skip that today)

In [ ]:
summary = np.array([0.0, 0.0, 0.0])
for word in keys:
    summary += keys[word] * attention[word]
summary

<div style="text-align: center; margin: 30px 0;">
  <img src="graphics/transformers_attention_details4.svg" style="max-width: 800px; max-height: 600px; border: 2px solid #ddd; border-radius: 8px; padding: 5px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
  <div style="margin-top: 10px; font-style: italic; color: #666; font-size: 0.9em;">
    Attention mechanism details (step 4)
  </div>
</div>

### Calculate the final prediction

We now add or concatenate our sentence summary with the embedding of the first word in
the English sentence. 

On top we apply some neural network layers (dense / fully connected layers) to predict the next word

In [ ]:
def prediction_head(x):
    # return model(x) # usually would do this
    return x  # today let's just do this

predicted = prediction_head([query + summary])
predicted

<div style="text-align: center; margin: 30px 0;">
  <img src="graphics/transformers_attention_details5.svg" style="max-width: 800px; max-height: 600px; border: 2px solid #ddd; border-radius: 8px; padding: 5px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
  <div style="margin-top: 10px; font-style: italic; color: #666; font-size: 0.9em;">
    Attention mechanism details (step 5)
  </div>
</div>

## Self-attention

What we just saw was cross-attention. 

When we are doing text generation instead of translation, we typically use self-attention.

In self-attention, we calculate the attention vector with respect to the previous words in the sentence.

<div style="text-align: center; margin: 30px 0;">
  <img src="graphics/transformers_attention_details_self.svg" style="max-width: 800px; max-height: 600px; border: 2px solid #ddd; border-radius: 8px; padding: 5px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
  <div style="margin-top: 10px; font-style: italic; color: #666; font-size: 0.9em;">
    Self-attention mechanism
  </div>
</div>

## Transformer architecture

This is the decoder-only version used in basically all current LLMs. 

It only uses self-attention (in orange), no cross-attention.

<div style="text-align: center; margin: 30px 0;">
  <img src="graphics/transformers_architecture_numbered.png" style="max-width: 300px; max-height: 700px; border: 2px solid #ddd; border-radius: 8px; padding: 5px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
  <div style="margin-top: 10px; font-style: italic; color: #666; font-size: 0.9em;">
    Decoder-only transformer architecture (numbered components) — adapted from Figure 1 of Vaswani et al., <a href="https://arxiv.org/abs/1706.03762">"Attention Is All You Need"</a> (2017)
  </div>
</div>

This architecture contains some details we have not yet discussed

1. The attention block is repeated N times. The first attention block takes in the input embeddings, 
   the second attention block takes in the output embeddings of the first attention block, etc.

2. Multi-head attention = in each attention block, we perform the attention lculation from above multiple times, 
    with different weight matrices. Intuitively, one attention head might focus on putting attention to grammatical concepts,
   another one on semantic meaning. In reality, the different heads are not as clearly separated, 
   each head does a little bit of everything.
   
3. With the method described above, the position of the word in the sentence is irrelevant to the model. 
   i.e. for the model "I went by bike" is the same as "by went bike I". To fix that, information about the
   position of the word is added to the input embeddings.
   
4. Layer normalisation to normalise the input to the layer to some mean and standard deviation. 
   This makes model training more stable and allows for training larger models. Today, usually the Normalisation
   is placed before the Attention and Feed Forward layers (rather than after as displayed here)



## Advantages of Transformers compared to Recurrent Neural Networks

Until 2017, [📙 **Recurrent Neural Networks (RNNs)**](../dl/dl-recurrent-neural-networks.ipynb) were the state of the art for language processing with deep learning, e.g. machine translation. The transformer architecture provides several advantages that have helped to overcome the limitations of RNNs and train much more capable models.

### 1. Access to information

The RNN only has direct access to the most recent word and to the overall summary of 
    the sentence as encoded in its current state. It <b> can not perform its own weighting</b> of
what words could be important for the next prediction.

<div style="text-align: center; margin: 30px 0;">
  <img src="graphics/transformers_versus_rnn.svg" style="max-width: 700px; max-height: 500px; border: 2px solid #ddd; border-radius: 8px; padding: 5px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
  <div style="margin-top: 10px; font-style: italic; color: #666; font-size: 0.9em;">
    Transformers vs RNNs comparison
  </div>
</div>


### 2. Parallel instead of recursive

The RNN always needs to be run in a recursive manner. Predict one word, feed back into the
model, etc. 
    
In contrast, the transformer architecture can be trained on multiple timesteps at the same
time. For example, we can feed the transformer the following samples all at the same time:

```    
[Ich bin -> mit] 
[Ich bin mit -> dem] 
[Ich bin mit dem -> Fahrrad]
``` 

Since the whole attention mechanism is easily parallellised using matrix multiplications,
this means that a transformer model can learn from all of these samples at the same time.

This greatly speeds up model training and also makes it much more stable, allowing to
train much larger models.


Of course, during prediction we again need to go step-by-step.




## Sources

- _The visualisations take heavy inspiration from [Prof. Herman Kamper](https://www.kamperh.com/nlp817)_

---
_This notebook is licensed under a [Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International (CC BY-NC-SA 4.0)](https://creativecommons.org/licenses/by-nc-sa/4.0/). Copyright © 2024-2026 [Christian Staudt](https://clstaudt.me) and [Katharina Rasch](https://krasch.io)_